# 🚀 **LoRA Fine-Tuning Assignment**

In this assignment, you will learn how to fine-tune a small language model using **LoRA** (Low-Rank Adaptation) on Google Colab with limited GPU resources.
We will also evaluate model training using **perplexity score**, and run ablation studies to understand the impact of hyperparameters.

**You will:**

1.  Perform supervised fine-tuning of `HuggingFaceTB/SmolLM-135M` using LoRA.
2.  Complete the provided Colab notebook.
3.  Submit both the code and a PDF report.

**The report must include:**

-   A graph showing perplexity across training checkpoints.
-   Specific examples of text generated by the model *before* and *after* fine-tuning for given prompts.
-   Discussion of any observed impact of hyperparameter choices on training and generated text.
- The training loss graph from `wandb`


## 📚 **Background**
### What is LoRA?
**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning method that injects trainable low-rank matrices into frozen pre-trained models. It works by modifying only a small subset of parameters while keeping most of the model fixed. This significantly reduces the memory and compute requirements during training.

*Key Concept:*
- Instead of fine-tuning full weight matrices, LoRA learns low-rank updates (A, B) such that W ≈ W₀ + BA



### What is Perplexity?
**Perplexity** is a measure of how well a language model predicts a sample of text. It evaluates the model’s ability to assign high probability to the correct next word in a sequence. Lower perplexity indicates better performance, meaning the model is less “perplexed” by the text.

Mathematically, perplexity is calculated as the exponentiation of the average negative log-likelihood loss:
$$
\text{Perplexity} = \exp(\text{Cross Entropy Loss})
$$

Perplexity is widely used during pretraining and fine-tuning of language models as a way to monitor fluency and general language modeling capability.


---
## **Step 1: Install & Load Packages**
For the first step, we install the necessary packages. For this project we will be using:
-  transformers: For loading pretrained language models, tokenizers, and training configurations.
-  peft: Enables parameter-efficient fine-tuning techniques like LoRA to reduce training cost.
- datasets: Provides easy access to public datasets from Huggingface.
- trl: Simplifies SFT and other tuning methods for language models

In [ ]:
!pip install datasets trl

**If you encounter such a message as shown below, restart the runtime and continue from below cells**

**If you encounter a CUDA/device mismatch error, restart the runtime and continue from the cells below.**


In [ ]:
# import the necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig


---

## **Step 2: Set Up Weights & Biases (Wandb)**  
Before starting the experiment, you are recommended to create a free account on [Weights & Biases (Wandb)](https://wandb.ai/site/), a platform for tracking and visualizing training metrics.  

Once you've signed up, go to [wandb.ai/authorize](https://wandb.ai/authorize) to get your API key.  

After obtaining the API key, you can login to your account with the code below. When you do training, your training process will be logged in there (Link to the dashboard will be provided at the very beginning of the training).




In [ ]:
import wandb
wandb.login()

---
## **Step 3: Load models and define LoRA Config**
For the second step, you load a model as well its corresponding tokenizer from huggingface, then set up the LoRA configuration.

- Here, we will be using [SmolLM-135M](https://huggingface.co/HuggingFaceTB/SmolLM-135M), a lightweight language model.



In [ ]:
# Load pre-trained model and tokenizer. This is a light-weight model good for running on Colab
model_name = "HuggingFaceTB/SmolLM-135M"

# Use the AutoModelForCausalLM, AutoTokenizer utilities to instantiate model and tokenizer.
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # for padding and adding towards the end


Now, you set up LoRA configuration.

**Hyperparameters:**

- **r**            : Rank of the low-rank matrices A and B. Higher values increase capacity but also computation.
- **lora_alpha**   : Scaling factor α applied to the LoRA weights during the forward pass. Controls the update magnitude.
- **target_modules**: List of module names (typically linear layers in the transformer) where LoRA is injected.
- **lora_dropout** : Dropout applied on the LoRA branch to regularize and prevent overfitting.
- **bias**         : Controls whether to train bias terms. "none" ignores all biases; "all" trains all biases; "lora_only" trains biases only in modules where LoRA is applied.
- **task_type**    : Specifies the downstream task. Here we use "CAUSAL_LM" for causal language modeling (e.g., autoregressive generation).



In [ ]:
# Function to generate text based on a prompt
def generate_response(model, tokenizer, prompt, max_length=100, num_return_sequences=1):

    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=0.7,  # Controls randomness (higher = more random)
        top_p=0.9,        # Nucleus sampling parameter
        do_sample=True,   # Use sampling instead of greedy decoding
    )

    # Decode and return the generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts[0]

**Performing inference before fine-tunning**

In [ ]:
# Baseline behaviour before fine-tunning
baseline_prompts = [
    "> Embark on a global treasure hunt\n> Decode cryptic clues\n",
    "> Work as a cashier\n> Customer counts their change\n > Realize the importance of patience \n >",
    "> Explore the ruins of Pompeii\n> Uncover ancient artifacts\n>\n",
    "> Take on the role of a homeless person\n> Experience the hardships and stigma\n> Learn the true meaning of resilience\n>\n",
]

# Move model to GPU for generation
model.to("cuda")

print("=" * 60)
print("BEFORE FINE-TUNING (base SmolLM-135M)")
print("=" * 60)
before_outputs = {}
for p in baseline_prompts:
    out = generate_response(model, tokenizer, p)
    before_outputs[p] = out
    print(out)
    print("-" * 60)

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r= 16,  # Rank of the low-rank matrices (higher = more capacity, but more compute)
    lora_alpha= 32,  # Scaling factor for the LoRA weights (α)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","up_proj", "down_proj", "gate_proj"],  # Modules inside the transformer layers to inject LoRA adapters into
    lora_dropout= 0.05,  # Dropout rate for LoRA layers to prevent overfitting
    bias="none",  # Whether to train bias terms ("none", "all", or "lora_only")
    task_type="CAUSAL_LM"  # Task type: causal language modeling (CLM)
)


# Apply LoRA to the model
model = get_peft_model(model, lora_config)

---
## **Step 4: Load and configure Dataset**
In this step, you
1. load the dataset for SFT.
  - Here, we will be loading datasets from Huggingface.
2. preprocessing the dataset as instructed below.

Specifically, you can choose one of these Greentexts dataset:

1. [maxmyn/wholesome_greentext_110k](https://huggingface.co/datasets/maxmyn/wholesome_greentext_110k)
2. [maxmyn/thank_you_greentext](https://huggingface.co/datasets/maxmyn/thank_you_greentext)
3. [maxmyn/wholesome_simple_greentext_133k](https://huggingface.co/datasets/maxmyn/wholesome_simple_greentext_133k)

In [ ]:
dataset_name = "maxmyn/wholesome_greentext_110k"

train_dataset = load_dataset(dataset_name, split="train[:100]") # We only use the first 100 rows in the dataset for training & evaluation.

In [ ]:
train_dataset.info

In [ ]:
# Connect the drive
from google.colab import drive
drive.mount('/content/drive')



## **Step 4: Training Setup**
In this step, you will need to:
- **Configure the SFT settings**, including training arguments like batch size and number of epochs
- **Set up the `SFTTrainer`** to fine-tune your model using the LoRA adapters

*You can find resources on SFTTrainer and SFTConfig [here](https://huggingface.co/docs/trl/en/sft_trainer). *You can check out the parameters available to you from this link.*

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/AI_ML_HW5_Checkpoints",  # Directory to save model checkpoints
    per_device_train_batch_size=4,                           # Number of samples per batch on each device (GPU).
    save_steps=100,                                          # Save a checkpoint every N steps. Larger values save less frequently and use less storage.

    learning_rate=2e-4,                          # Set your learning rate (e.g., 2e-4). Controls how fast the model updates.
    dataset_text_field="greentexts",                     # Name of the dataset column containing the input text (e.g., "text" or "prompt").

    logging_steps=1,                                         # Log metrics every N steps — useful for tracking training progress.
    run_name="SmolLM-135M - wholesome_greentext_110k",        # Name of the run on Wandb for experiment tracking.
    max_steps=500,                                           # Total number of training steps. Adjust based on available compute — ~500 steps usually takes 20–25 mins.
    # num_train_epochs=3,                                    # Number of times to iterate over the full training dataset.
    report_to="wandb",
                                                            # Ignored if max_steps is set — in that case, training stops after reaching max_steps regardless of epochs.
                                                             # If you leave max_steps unset, it will determine the max step size based on dataset size and batch size.
    fp16=True,
    bf16=False,
)

In [ ]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset
)

**Now, you run the trainer.**

In [ ]:
trainer.train()

---
## **Step 5: Evaluate**

After done with training. you evaluate the performance of the model over different checkpoints.

-   A graph showing perplexity across training checkpoints.
-   Specific examples of text generated by the model *before* and *after* fine-tuning for given prompts.


*Note: Loading multiple models simultaneously may lead to memory issues. Please manage resources carefully to avoid potential crashes.*

### Step 5.1. Measure perplexity

Here, you will use the `SFTTrainer` tool to calculate perplexity. To see the quality of the training, we use the same dataset that we used for training for evaluation.


In [ ]:
# Load the model
model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/AI_ML_HW5_Checkpoints/checkpoint-500")

# Define the trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset
)

# check evaluation result. Here. the 'eval_loss' represents the cross entropy loss
eval_results = trainer.evaluate()

In [ ]:
# Implement the function for computing perplexity. Feel free to import packages like 'math'
import math
def calculate_perplexity(log_loss):
    perplexity = math.exp(log_loss)
    return perplexity


In [ ]:
import wandb, math, matplotlib.pyplot as plt

api = wandb.Api()
run = api.run("cb5330-new-york-university/huggingface/my_first_lora_run")
hist = run.history(keys=["train/loss"])

s16 = hist["_step"].tolist()
l16 = hist["train/loss"].tolist()
ppl16 = [math.exp(x) for x in l16]

# --- Print perplexity at checkpoint steps ---
def ppl_at_step(steps, ppls, target):
    if not steps:
        return None
    idx = min(range(len(steps)), key=lambda i: abs(steps[i] - target))
    return ppls[idx]

checkpoints = [0, 100, 200, 300, 400, 500]
print(f"{'Step':>6} | {'r=16 PPL':>12}")
print("-" * 23)
for t in checkpoints:
    p = ppl_at_step(s16, ppl16, t)
    print(f"{t:>6} | {p:>12.4f}")
print()

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(s16, ppl16, color='#3d5a80', linewidth=2)
ax.set_xlabel('Training step')
ax.set_ylabel('Perplexity')
ax.set_yscale('log')
ax.set_title('Training Perplexity — LoRA r=16')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 5.2. Give specific examples of generated text.

Next, you will demonstrate the impact of fine-tuning by generating text with the model *before* and *after* training.

First, implement the text generation function. Then, use a selection of prompts derived from the training dataset to generate text with both the base model and your fine-tuned LoRA model.


  **Example of Prompt and text generation:**

    
    greentexts:
    > Imagine being able to fly
    > Picture yourself soaring through the sky
    > Realize you're afraid of heights
    > Go back to play video games

    Prompt:
    > Imagine being able to fly
    > Picture yourself soaring through the sky
    > Realize you're afraid of heights
    >

    Generated (ideally):
    Go back to play video games
    

Check the output

In [ ]:
# generate the same prompts before fine-tunning after fine tunning
# Baseline behaviour before fine-tunning
print("=" * 60)
print("After FINE-TUNING (base SmolLM-135M + LoRA)")
print("=" * 60)
after_outputs = {}
for p in baseline_prompts:
    out = generate_response(model, tokenizer, p)
    after_outputs[p] = out
    print(out)
    print("-" * 60)

**Remarks**

*Positives*
1. The model has learned the strucutre and gives a sentence in a structured manner.


*Negatives*

1. Weak semantic coherence between an example.
2. Heavy repetition within the sentences.

---
## **Step 6: Ablation Studies**

Here, you conduct at least one set of ablation study. Repeat the experiment with different settings. It includes but not limited to:
- **Different model** (we recommend `openai-community/gpt2`, which is another lightweight model that T4 chip can handle)
- **Different dataset** (you can either use another greentext dataset or find your own dataset)
- **LoRA layer number**
- **LoRA dropout ratio**
- **Learning rate**
- ...

Please complete the code and share the result in the report as well.



The Ablation Study will include changing the LoRA layer number from r = 16 to r = 4 and analysing how this changes the running time, perplexity, and inference results.

In [ ]:
# Ablation: LoRA rank r=4 instead of r=16

# Define the new model
model_ab = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")

# Define the configurations for LoRA when r = 4
lora_cfg_r4 = LoraConfig(
    r=4, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

# Apply LoRA to the model
model_ab = get_peft_model(model_ab, lora_cfg_r4)

# Defining the parameters for fine tunning
args_ab = SFTConfig(
    output_dir="/content/drive/MyDrive/AI_ML_HW5_Checkpoints/smollm_r4",
    per_device_train_batch_size=4, save_steps=100, learning_rate=2e-4,
    dataset_text_field="greentexts", logging_steps=1,
    run_name="SmolLM-135M r=4", max_steps=500,
    report_to="wandb",
)

# Define the trainer
trainer_ab = SFTTrainer(model=model_ab, args=args_ab,
                       train_dataset=train_dataset, eval_dataset=train_dataset)

# Perform the training
trainer_ab.train()

# Perform the evaluation of the trainer
res_ab = trainer_ab.evaluate()

Looking at the baseline prompts after fine-tunning and changing the LoRA layer number from 16 to 4

In [ ]:
# generate the same prompts before fine-tunning after fine tunning
# Baseline behaviour before fine-tunning
print("=" * 60)
print("After FINE-TUNING Ablation (base SmolLM-135M + LoRA)")
print("=" * 60)
after_outputs = {}
for p in baseline_prompts:
    out = generate_response(model_ab, tokenizer, p)
    after_outputs[p] = out
    print(out)
    print("-" * 60)

Comparison of perplexity graphs

In [ ]:
import math, matplotlib.pyplot as plt

def loss_curve(trainer):
    steps, losses = [], []
    for entry in trainer.state.log_history:
        if 'loss' in entry and 'eval_loss' not in entry:
            steps.append(entry['step'])
            losses.append(entry['loss'])
    return steps, losses

s4, l4 = loss_curve(trainer_ab)
ppl4 = [math.exp(x) for x in l4]

# --- Print perplexity at checkpoint steps ---
def ppl_at_step(steps, ppls, target):
    if not steps:
        return None
    idx = min(range(len(steps)), key=lambda i: abs(steps[i] - target))
    return ppls[idx]

checkpoints = [0, 100, 200, 300, 400, 500]
print(f"{'Step':>6} | {'r=4 PPL':>12}")
print("-" * 23)
for t in checkpoints:
    p = ppl_at_step(s4, ppl4, t)
    print(f"{t:>6} | {p:>12.4f}")
print()

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(s4, ppl4, color='#e07a5f', linewidth=2)
ax.set_xlabel('Training step')
ax.set_ylabel('Perplexity')
ax.set_yscale('log')
ax.set_title('Training Perplexity — LoRA r=4')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()